In [1]:
%pip install -q -U google-genai termcolor google-cloud-bigquery db-dtypes thefuzz jsonref google-cloud-aiplatform

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
PROJECT_ID = "ghp-poc"
REGION = "us-central1"

DATA_AXLE = "ghp-poc.jobseq.data_axle"
INDUSTRY_2024Q2 = "ghp-poc.jobseq.industry_2024q2"
OCCUPATION_2024Q2 = "ghp-poc.jobseq.occupation_2024q2"
WAGES_2024Q2 = "ghp-poc.jobseq.wages_2024q2"

MAX_STORE_RESULTS = 10
MAX_PRODUCT_RESULTS = 10
STORE_NAME_SIMILARITY_THRESHOLD = 90
PRODUCT_SEARCH_SIMILARITY_THRESHOLD = 80

### Dependency Imports

In [3]:
from typing import Optional

import pydantic
import termcolor
import pandas as pd
from thefuzz import fuzz
from google import genai
from google.genai import types as genai_types
from google.cloud import bigquery

### Initialize Clients

In [4]:
bq_client = bigquery.Client(project=PROJECT_ID)
genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

### Pydantic Models

In [5]:
# DB Models
class MetroMatrix(pydantic.BaseModel):
    City: Optional[str]
    State: Optional[str]
    County: Optional[str]

class HQRelocation(pydantic.BaseModel):
    City: Optional[str]
    State: Optional[str]
    County: Optional[str]
    Industry: Optional[str]

class CompanyRelocation(pydantic.BaseModel):
    City: Optional[str]
    State: Optional[str]
    County: Optional[str]
    Industry: Optional[str]

# Tool return types
class MetroMatrixResult(pydantic.BaseModel):
    city_analysis: list[MetroMatrix] = []
    error: Optional[str] = None

class HQRelocationResult(pydantic.BaseModel):
    city_analysis: list[HQRelocation] = []
    error: Optional[str] = None

class CompanyRelocationResult(pydantic.BaseModel):
    city_analysis: list[CompanyRelocation] = []
    error: Optional[str] = None

## Tools

user journeys:
1. Metro Matrix request
    - City names
    - State names
2. Head Quarter Relocation
    - City name
    - State name
    - Industry name
3. Company Relocation
    - City name
    - State name
    - Industry name

### Metro Matrix 

In [6]:
def find_metro_matrix(
    city_name: str,
    state_name: Optional[str]=None,
) -> MetroMatrixResult:
    f"""Search for overall metro data for the specified city. Called for each city in the MetroMatrix comparison.

    Args:
        city_name (str): The name of the city.
        state_name (Optional, str): The 2-letter abbreviation of the state specified by the user.

    Returns:
        MetroMatrixResult: The return value. Object including overall data for the city, state.
    """

    # Initialize query parameters
    query_parameters = list[
        bigquery.ScalarQueryParameter | bigquery.ArrayQueryParameter
    ]()

    # probably convert city and state into some kind of number ##
    city_selector = None
    if city_name:
        city_selector = f"req_city.City = '{city_name}'"
        query_parameters.extend(
        [
            bigquery.ScalarQueryParameter(
                name="city_name",
                type_=bigquery.SqlParameterScalarTypes.STRING,
                value=city_name,
            ),
        ])
        if state_name:
            city_selector = f"req_city.City = '{city_name}' AND req_city.State = '{state_name}'"
            query_parameters.extend(
                [
                    bigquery.ScalarQueryParameter(
                        name="state_name",
                        type_=bigquery.SqlParameterScalarTypes.STRING,
                        value=state_name,
                    ),
                ]
            )
        
    where_clause = ""
    if city_selector:
        where_clause = f"WHERE {city_selector}"

    select_city_query = f"SELECT * FROM {DATA_AXLE}"
    
    # TODO: UPDATE THIS - just test

    query = f"""
SELECT
    req_city.City,
    req_city.State,
    req_city.County
FROM
    ({select_city_query}) AS req_city
{where_clause} LIMIT 1
""".strip()
    print(query)

    query_job_config = bigquery.QueryJobConfig()
    query_job_config.query_parameters = query_parameters

    query_job = bq_client.query(
        query=query,
        job_config=query_job_config,
    )

    query_df: pd.DataFrame = query_job.to_dataframe()
    print("Dataframe: "+query_df)

    city_analysis = [
        MetroMatrix.model_validate(row.to_dict())
         for idx, row in query_df.iterrows()]
    print(city_analysis)

    return MetroMatrixResult(city_analysis=city_analysis)

In [7]:
def find_hq_relocation(
    city_name: str,
    industry: Optional[str]=None,
    state_name: Optional[str]=None,
) -> HQRelocationResult:
    f"""Search for Company Headquarters Relocation Data.

    Args:
        city_name (str): The name of the city.
        state_name (str): The 2-letter abbreviation of the state specified by the user.
        industry (str): The industry of the comapny considering Head Quarter relocation.

    Returns:
        HQRelocationResult: The return value. Object including overall industry data for the city, state.
    """

    # Initialize query parameters
    query_parameters = list[
        bigquery.ScalarQueryParameter | bigquery.ArrayQueryParameter
    ]()

    # probably convert city and state into some kind of number ##
    city_selector = None
    if city_name:
        city_selector = f"req_city.City = '{city_name}'"
        query_parameters.extend(
        [
            bigquery.ScalarQueryParameter(
                name="city_name",
                type_=bigquery.SqlParameterScalarTypes.STRING,
                value=city_name,
            ),
        ])
        if state_name:
            city_selector = f"req_city.City = '{city_name}' AND req_city.State = '{state_name}'"
            query_parameters.extend(
                [
                    bigquery.ScalarQueryParameter(
                        name="state_name",
                        type_=bigquery.SqlParameterScalarTypes.STRING,
                        value=state_name,
                    ),
                ]
            )
        
    where_clause = ""
    if city_selector:
        where_clause = f"WHERE {city_selector}"

    select_city_query = f"SELECT * FROM {DATA_AXLE}"
    
    # TODO: UPDATE THIS - just test

    query = f"""
SELECT
    req_city.City,
    req_city.State,
    req_city.County,
    req_city.`Primary SIC Description` as Industry
FROM
    ({select_city_query}) AS req_city
{where_clause} LIMIT 1
""".strip()
    print(query)

    query_job_config = bigquery.QueryJobConfig()
    query_job_config.query_parameters = query_parameters

    query_job = bq_client.query(
        query=query,
        job_config=query_job_config,
    )

    query_df: pd.DataFrame = query_job.to_dataframe()
    print("Dataframe: "+query_df)

    city_analysis = [
        HQRelocation.model_validate(row.to_dict())
         for idx, row in query_df.iterrows()]
    print(city_analysis)

    return HQRelocationResult(city_analysis=city_analysis)

In [8]:
def find_company_relocation(
    city_name: str,
    industry: Optional[str]=None,
    state_name: Optional[str]=None,
) -> CompanyRelocationResult:
    f"""Search for Company Relocation Data.

    Args:
        city_name (str): The name of the city.
        state_name (str): The 2-letter abbreviation of the state specified by the user.
        industry (str): The industry of the comapny considering relocation.

    Returns:
        CompanyRelocationResult: The return value. Object including overall industry data for the city, state.
    """

    # Initialize query parameters
    query_parameters = list[
        bigquery.ScalarQueryParameter | bigquery.ArrayQueryParameter
    ]()

    # probably convert city and state into some kind of number ##
    city_selector = None
    if city_name:
        city_selector = f"req_city.City = '{city_name}'"
        query_parameters.extend(
        [
            bigquery.ScalarQueryParameter(
                name="city_name",
                type_=bigquery.SqlParameterScalarTypes.STRING,
                value=city_name,
            ),
        ])
        if state_name:
            city_selector = f"req_city.City = '{city_name}' AND req_city.State = '{state_name}'"
            query_parameters.extend(
                [
                    bigquery.ScalarQueryParameter(
                        name="state_name",
                        type_=bigquery.SqlParameterScalarTypes.STRING,
                        value=state_name,
                    ),
                ]
            )
        
    where_clause = ""
    if city_selector:
        where_clause = f"WHERE {city_selector}"

    select_city_query = f"SELECT * FROM {DATA_AXLE}"
    
    # TODO: UPDATE THIS - just test

    query = f"""
SELECT
    req_city.City,
    req_city.State,
    req_city.County,
    req_city.`Primary SIC Description` as Industry
FROM
    ({select_city_query}) AS req_city
{where_clause} LIMIT 1
""".strip()
    print(query)

    query_job_config = bigquery.QueryJobConfig()
    query_job_config.query_parameters = query_parameters

    query_job = bq_client.query(
        query=query,
        job_config=query_job_config,
    )

    query_df: pd.DataFrame = query_job.to_dataframe()
    print("Dataframe: "+query_df)

    city_analysis = [
        CompanyRelocation.model_validate(row.to_dict())
         for idx, row in query_df.iterrows()]
    print(city_analysis)

    return CompanyRelocationResult(city_analysis=city_analysis)

In [9]:
def invalid_request(
    user_request: str,
) -> str:
    f"""Respond to customer and help clarify thier question.

    Args:
        user_request (str): The request from the user.

    Returns:
        str: The response to the user
    """
    return "I am only able to generate Metro Matrix, HQ Relocation, and Company Relocation reports at the moment. Can I help you create one of those?"

## Test Agent

In [10]:
def print_content(content: genai_types.Content):
    for part in content.parts:
        if part.function_call:
            color = "cyan"

            fn_name = part.function_call.name or "unknown"
            fn_args = part.function_call.args or {}

            fn_args_string = ", ".join(f"{k}={v}" for k, v in fn_args.items())
            fn_string = f"{fn_name}({fn_args_string}) => "

            print(termcolor.colored(text=fn_string, color=color), end="")

        elif part.function_response:
            color = "cyan"

            fn_name = part.function_response.name

            response_dict = part.function_response.response

            print(termcolor.colored(text=response_dict, color=color))

        elif part.text:
            color = "white" if content.role == "user" else "green"

            print(termcolor.colored(text=part.text, color=color))

        else:
            color = "red"

            print(termcolor.colored(text=part, color=color))

### Create Function Calling Agent

In [11]:
# Setup Agent System Instructions
system_instruction = """
You are an Economic Analyst. Your primary function is to help users create reports related to metro matrix city comparison, company headquarters relocation, and company relocation. You have access to the following tools:
-  find_metro_matrix: Use this when the user requests a MetroMatrix Report. This tools should be called for each city requested as part of the metro matrix comparison.
-  find_hq_relocation:  Use this to find data about companies moving their headquarters.
-  find_company_relocation: Use this to find data about companies relocating their offices or facilities.
If a request is unrelated to these functions, use invalid_request.

When the user asks for information about a location, they must provide the city name. If the user specified a state, use the state's two letter abreviation for the function call. If the query requires an industry, clarify the industry if needed.

If a user request is ambiguous, ask clarifying questions to determine the exact function and parameters needed. Do not make assumptions about missing information.
"""

### Test Function Calling Agent

In [13]:
response = genai_client.models.generate_content(
    model="gemini-1.5-flash-002",
    contents="help me create a metro matrix for Houston and Sugar Land",
    # contents="create a hq relocation report for a manufacturing company moving to houston",
    # contents="I want to create a company relocation report for a manufacturing company moving to Houston",
    # contents="Can you help me create a specialized city comparison report",
    config=genai_types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.2,
        candidate_count=1,
        seed=42,
        tools=[find_metro_matrix, find_hq_relocation, find_company_relocation, invalid_request],
    ),
)

history = response.automatic_function_calling_history + [response.candidates[0].content]

for content in history:
    print_content(content)

SELECT
    req_city.City,
    req_city.State,
    req_city.County
FROM
    (SELECT * FROM ghp-poc.jobseq.data_axle) AS req_city
WHERE req_city.City = 'Houston' LIMIT 1


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1900: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


                 City          State             County
0  Dataframe: Houston  Dataframe: TX  Dataframe: Harris
[MetroMatrix(City='Houston', State='TX', County='Harris')]
SELECT
    req_city.City,
    req_city.State,
    req_city.County
FROM
    (SELECT * FROM ghp-poc.jobseq.data_axle) AS req_city
WHERE req_city.City = 'Sugar Land' LIMIT 1


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1900: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


                    City          State                County
0  Dataframe: Sugar Land  Dataframe: TX  Dataframe: Fort Bend
[MetroMatrix(City='Sugar Land', State='TX', County='Fort Bend')]
help me create a metro matrix for Houston and Sugar Land
find_metro_matrix(city_name=Houston) => find_metro_matrix(city_name=Sugar Land) => {'result': MetroMatrixResult(city_analysis=[MetroMatrix(City='Houston', State='TX', County='Harris')], error=None)}
{'result': MetroMatrixResult(city_analysis=[MetroMatrix(City='Sugar Land', State='TX', County='Fort Bend')], error=None)}
Here is a Metro Matrix Report comparing Houston and Sugar Land, Texas.

Houston:
County: Harris
State: TX

Sugar Land:
County: Fort Bend
State: TX

Please let me know if you would like any additional information.

